# 01 — Inspect TinyStories-3M
Inspect the pinned PyTorch model's architecture, parameter counts, and tied embeddings.
Each fresh execution writes `model_inspection.json` into a unique `results/notebook_runs/<run-id>/`
folder, alongside source snapshots and `run.json`. Initial loading requires a model cache or download.

Start from a fresh kernel and run top to bottom. The final cell records output hashes; incomplete runs remain `started`.


In [ ]:
from pathlib import Path
import sys

# Works when Jupyter starts in the repository root or notebooks/.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/python/workflow_config.py').is_file()), None)
if ROOT is None:
    raise RuntimeError('Start Jupyter in the extracted tinystories-rv64imc folder.')
sys.path.insert(0, str(ROOT / 'src/python'))
from workflow_config import (MODEL_ID, MODEL_REVISION, SEQUENCE_LENGTH,
                             MODEL_DIR, EXPORT_DIR, RESULT_DIR, C_DIR, MODEL_PATH,
                             PROMPT, GENERATED_LENGTH)
from notebook_runs import start_run, finish_run
RUN_DIR = start_run(ROOT, 'notebooks/01_inspect_model.ipynb', {'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'sequence_length': SEQUENCE_LENGTH})
RESULT_DIR = RUN_DIR


In [ ]:
from collections import defaultdict

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

## Load the pinned checkpoint

In [ ]:
print(f"PyTorch version: {torch.__version__}")
print(f"Loading {MODEL_ID}...")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
)

In [ ]:
model.eval();
print(model.transformer.h[0])

## Architecture and storage

In [ ]:
# model.parameters() counts independent Parameter objects.
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
)

trainable_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

parameter_bytes = sum(
    parameter.numel() * parameter.element_size()
    for parameter in model.parameters()
)

In [ ]:
print("\n=== Model type ===")
print(type(model).__name__)

In [ ]:
print("\n=== Configuration ===")
print(f"Vocabulary size        : {model.config.vocab_size:,}")
print(f"Maximum sequence length: {model.config.max_position_embeddings:,}")
print(f"Transformer layers     : {model.config.num_layers}")
print(f"Hidden dimension       : {model.config.hidden_size}")
print(f"Attention heads        : {model.config.num_heads}")
print(f"Attention types        : {model.config.attention_layers}")
print(f"Activation function    : {model.config.activation_function}")

In [ ]:
print("\n=== Parameter count ===")
print(f"Total parameters       : {total_parameters:,}")
print(f"Trainable parameters   : {trainable_parameters:,}")
print(f"FP32 parameter storage : {parameter_bytes / (1024**2):.2f} MiB")
print(f"Estimated INT8 storage : {total_parameters / (1024**2):.2f} MiB")

In [ ]:
# Count parameters by major model component.
component_counts = defaultdict(int)

for name, parameter in model.named_parameters():
    parts = name.split(".")

    if len(parts) >= 3 and parts[0] == "transformer" and parts[1] == "h":
        component = f"transformer_block_{parts[2]}"
    elif name.startswith("transformer.wte"):
        component = "token_embedding"
    elif name.startswith("transformer.wpe"):
        component = "position_embedding"
    elif name.startswith("transformer.ln_f"):
        component = "final_layer_norm"
    elif name.startswith("lm_head"):
        component = "language_model_head"
    else:
        component = "other"

    component_counts[component] += parameter.numel()

In [ ]:
print("\n=== Parameters by component ===")

for component, count in component_counts.items():
    percentage = 100.0 * count / total_parameters
    print(f"{component:24s}: {count:10,d} ({percentage:6.2f}%)")

In [ ]:
# GPT-style models commonly share the token-embedding matrix with
# the output language-model head.
input_embedding = model.get_input_embeddings().weight
output_embedding = model.get_output_embeddings().weight

weights_are_tied = (
    input_embedding.data_ptr() == output_embedding.data_ptr()
)

print("\n=== Weight sharing ===")
print(f"Input/output embeddings tied: {weights_are_tied}")

## Save a compact result

In [ ]:
import json
RESULT_DIR.mkdir(parents=True, exist_ok=True)
report = {
    'model_id': MODEL_ID, 'model_revision': MODEL_REVISION,
    'total_parameters': total_parameters, 'trainable_parameters': trainable_parameters,
    'parameter_bytes': parameter_bytes, 'components': dict(component_counts),
    'embeddings_tied': weights_are_tied, 'config': model.config.to_dict(),
}
path = RESULT_DIR / 'model_inspection.json'
path.write_text(json.dumps(report, indent=2))
print('Saved:', path)

In [ ]:
finish_run(ROOT, RUN_DIR, [RESULT_DIR / 'model_inspection.json'])
print('Saved run record:', RUN_DIR / 'run.json')
